输出特征图尺寸
输入：3×32×32，卷积核：16个，每个大小为3×5×5，padding=2，stride=2
输出高度 = floor((32 + 2×2 - 5)/2) + 1 = floor((32+4-5)/2)+1 = floor(31/2)+1 = 15+1 = 16
输出宽度 = 同样为16
输出通道数 = 16
所以特征图尺寸为 16 × 16 × 16（通道数×高×宽）。

单个输出通道一个像素的点乘次数
每个卷积核包含 3×5×5 = 75 个权重，每个权重与输入的一个像素相乘，故每次输出像素需要 75 次乘法操作。

In [21]:
import numpy as np

def max_pool2d_forward(X, pool_size=(2, 2), stride=2, padding=0):
    """
    手动实现二维最大池化前向传播
    X: 输入数组，形状 (N, C, H, W)
    pool_size: 池化窗口 (H_k, W_k)
    stride: 步幅
    padding: 填充
    """
    N, C, H, W = X.shape
    H_k, W_k = pool_size
    
    # 计算输出尺寸
    H_out = (H + 2 * padding - H_k) // stride + 1
    W_out = (W + 2 * padding - W_k) // stride + 1
    
    # 对输入进行填充（仅空间维度）
    if padding > 0:
        X_padded = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)), 
                          mode='constant', constant_values=-np.inf)
    else:
        X_padded = X
    
    out = np.zeros((N, C, H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            h_end = h_start + H_k
            w_start = j * stride
            w_end = w_start + W_k
            # 提取窗口并取最大值
            window = X_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))
    
    return out

# 示例测试
if __name__ == "__main__":
    np.random.seed(0)
    x = np.random.randn(2, 3, 32, 32)
    out = max_pool2d_forward(x, pool_size=(2, 2), stride=2, padding=0)
    print("输出尺寸:", out.shape)  # 期望 (2, 3, 16, 16)

输出尺寸: (2, 3, 16, 16)


一个5×5卷积层（无偏置）的参数量
输入通道C，输出通道C，核大小5×5 → 参数量 = C × C × 5 × 5 = 25C²。

两个串联3×3卷积层（无偏置）的总参数量
每层参数量 = C × C × 3 × 3 = 9C²，两层总和 = 18C²。

In [20]:
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块：一个普通卷积层 + 两个 1x1 卷积层，每个卷积后跟 ReLU
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 示例
if __name__ == "__main__":
    import torch
    nin = NiNBlock(3, 16, kernel_size=5, stride=1, padding=2)
    x = torch.randn(1, 3, 32, 32)
    y = nin(x)
    print("NiN输出尺寸:", y.shape)  # (1, 16, 32, 32)

NiN输出尺寸: torch.Size([1, 16, 32, 32])


批量归一化输出
输入：x = [2, 4, 6, 8]
均值 μ = 5，方差 σ² = 5，σ = √5
γ = 2，β = 1，ϵ = 0

归一化：
x
^
i
=
x
i
−
μ
σ
x
^
  
i
​
 = 
σ
x 
i
​
 −μ
​
 
输出：
y
i
=
γ
x
^
i
+
β
y 
i
​
 =γ 
x
^
  
i
​
 +β

计算结果（精确形式）：
y
1
=
1
−
6
5
y 
1
​
 =1− 
5
​
 
6
​
 
y
2
=
1
−
2
5
y 
2
​
 =1− 
5
​
 
2
​
 
y
3
=
1
+
2
5
y 
3
​
 =1+ 
5
​
 
2
​
 
y
4
=
1
+
6
5
y 
4
​
 =1+ 
5
​
 
6
​
 

数值近似：
y₁ ≈ -1.6833，y₂ ≈ 0.1056，y₃ ≈ 1.8944，y₄ ≈ 3.6833

In [19]:
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块：两个 3x3 卷积（+BN） + 残差连接
    use_1x1conv: 若为 True，则用 1x1 卷积调整输入维度
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

# 示例
if __name__ == "__main__":
    import torch
    # 输入输出通道相同，不需要 1x1 conv
    res_block1 = Residual(64, 64)
    x = torch.randn(1, 64, 32, 32)
    print("输出尺寸:", res_block1(x).shape)  # (1, 64, 32, 32)
    
    # 输入输出通道不同，需要用 1x1 conv 调整维度
    res_block2 = Residual(64, 128, use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 64, 32, 32)
    print("输出尺寸:", res_block2(x2).shape)  # (1, 128, 16, 16)

输出尺寸: torch.Size([1, 64, 32, 32])
输出尺寸: torch.Size([1, 128, 16, 16])


为什么底层学习率小，顶层学习率大？
底层特征（边缘、纹理等）在大型源数据集上已训练得很好，具有通用性。小学习率（或冻结）可以保留这些通用特征，避免在目标小数据集上过拟合或灾难性遗忘。顶层输出层需要学习新任务的类别区分，随机初始化或与源任务差异大，因此需要较大学习率快速适应。

目标数据集很小且与源数据集非常相似时的策略

冻结所有底层特征提取层（即前若干层），只训练顶层分类层（全连接层）。

使用非常小的学习率进行微调，或者只重新训练最后一层（线性分类器）。

可增加数据增广、权重衰减等正则化手段。

In [18]:
import torchvision.transforms as transforms

# 定义组合增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪，面积比例 0.08~1.0，然后缩放到 224x224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度，变化范围 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转换为 Tensor
    transforms.ToTensor()
])

# 测试（需要 PIL 图像）
if __name__ == "__main__":
    from PIL import Image
    # 用随机数据模拟一张图像
    img = Image.new('RGB', (256, 256), color='red')
    tensor_img = augmentation_pipeline(img)
    print("输出 Tensor 形状:", tensor_img.shape)  # torch.Size([3, 224, 224])

ImportError: cannot import name 'OrderedDict' from 'typing' (C:\Users\12224\Anaconda3\lib\typing.py)

IoU 计算
真实框 A = [10, 10, 50, 50]
预测框 B = [30, 30, 70, 70]

交集矩形：左上角 (max(10,30)=30, max(10,30)=30)，右下角 (min(50,70)=50, min(50,70)=50)
交集面积 = (50-30) × (50-30) = 20×20 = 400

A 面积 = (50-10)×(50-10) = 40×40 = 1600
B 面积 = (70-30)×(70-30) = 40×40 = 1600
并集面积 = 1600 + 1600 - 400 = 2800

IoU = 400 / 2800 = 1/7（约 0.142857）

In [17]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "a229c0bd",
   "metadata": {},
   "source": [
    "输出特征图尺寸\n",
    "输入：3×32×32，卷积核：16个，每个大小为3×5×5，padding=2，stride=2\n",
    "输出高度 = floor((32 + 2×2 - 5)/2) + 1 = floor((32+4-5)/2)+1 = floor(31/2)+1 = 15+1 = 16\n",
    "输出宽度 = 同样为16\n",
    "输出通道数 = 16\n",
    "所以特征图尺寸为 16 × 16 × 16（通道数×高×宽）。\n",
    "\n",
    "单个输出通道一个像素的点乘次数\n",
    "每个卷积核包含 3×5×5 = 75 个权重，每个权重与输入的一个像素相乘，故每次输出像素需要 75 次乘法操作。"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "b71d8170",
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "\n",
    "def max_pool2d(x, kernel_size, stride=None, padding=0):\n",
    "    \"\"\"\n",
    "    二维最大池化前向传播（手动实现）\n",
    "    x: 输入张量，形状 (N, C, H, W) 或 (C, H, W)\n",
    "    kernel_size: 池化窗口大小 (kh, kw) 或 int\n",
    "    stride: 步幅 (sh, sw) 或 int，默认等于 kernel_size\n",
    "    padding: 填充数量 (ph, pw) 或 int\n",
    "    \"\"\"\n",
    "    # 统一格式\n",
    "    if isinstance(kernel_size, int):\n",
    "        kh = kw = kernel_size\n",
    "    else:\n",
    "        kh, kw = kernel_size\n",
    "    if stride is None:\n",
    "        sh = sw = kh\n",
    "    elif isinstance(stride, int):\n",
    "        sh = sw = stride\n",
    "    else:\n",
    "        sh, sw = stride\n",
    "    if isinstance(padding, int):\n",
    "        ph = pw = padding\n",
    "    else:\n",
    "        ph, pw = padding\n",
    "\n",
    "    # 处理输入维度\n",
    "    if x.ndim == 3:\n",
    "        C, H, W = x.shape\n",
    "        N = 1\n",
    "        x = x.reshape(1, C, H, W)\n",
    "    else:\n",
    "        N, C, H, W = x.shape\n",
    "\n",
    "    # 填充\n",
    "    pad_h = H + 2 * ph\n",
    "    pad_w = W + 2 * pw\n",
    "    x_pad = np.pad(x, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=0)\n",
    "\n",
    "    # 计算输出尺寸\n",
    "    out_h = (pad_h - kh) // sh + 1\n",
    "    out_w = (pad_w - kw) // sw + 1\n",
    "    out = np.zeros((N, C, out_h, out_w))\n",
    "\n",
    "    # 滑动窗口计算最大值\n",
    "    for n in range(N):\n",
    "        for c in range(C):\n",
    "            for i in range(out_h):\n",
    "                for j in range(out_w):\n",
    "                    h_start = i * sh\n",
    "                    h_end = h_start + kh\n",
    "                    w_start = j * sw\n",
    "                    w_end = w_start + kw\n",
    "                    window = x_pad[n, c, h_start:h_end, w_start:w_end]\n",
    "                    out[n, c, i, j] = np.max(window)\n",
    "    return out.squeeze() if out.shape[0] == 1 else out"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "63fc35c2",
   "metadata": {},
   "source": [
    "一个5×5卷积层（无偏置）的参数量\n",
    "输入通道C，输出通道C，核大小5×5 → 参数量 = C × C × 5 × 5 = 25C²。\n",
    "\n",
    "两个串联3×3卷积层（无偏置）的总参数量\n",
    "每层参数量 = C × C × 3 × 3 = 9C²，两层总和 = 18C²。"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "1a00b89f",
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch.nn as nn\n",
    "\n",
    "class NiNBlock(nn.Sequential):\n",
    "    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):\n",
    "        super().__init__(\n",
    "            # 普通卷积层\n",
    "            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),\n",
    "            nn.ReLU(),\n",
    "            # 第一个1×1卷积\n",
    "            nn.Conv2d(out_channels, out_channels, kernel_size=1),\n",
    "            nn.ReLU(),\n",
    "            # 第二个1×1卷积\n",
    "            nn.Conv2d(out_channels, out_channels, kernel_size=1),\n",
    "            nn.ReLU()\n",
    "        )"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6b075b79",
   "metadata": {},
   "source": [
    "批量归一化输出\n",
    "输入：x = [2, 4, 6, 8]\n",
    "均值 μ = 5，方差 σ² = 5，σ = √5\n",
    "γ = 2，β = 1，ϵ = 0\n",
    "\n",
    "归一化：\n",
    "x\n",
    "^\n",
    "i\n",
    "=\n",
    "x\n",
    "i\n",
    "−\n",
    "μ\n",
    "σ\n",
    "x\n",
    "^\n",
    "  \n",
    "i\n",
    "​\n",
    " = \n",
    "σ\n",
    "x \n",
    "i\n",
    "​\n",
    " −μ\n",
    "​\n",
    " \n",
    "输出：\n",
    "y\n",
    "i\n",
    "=\n",
    "γ\n",
    "x\n",
    "^\n",
    "i\n",
    "+\n",
    "β\n",
    "y \n",
    "i\n",
    "​\n",
    " =γ \n",
    "x\n",
    "^\n",
    "  \n",
    "i\n",
    "​\n",
    " +β\n",
    "\n",
    "计算结果（精确形式）：\n",
    "y\n",
    "1\n",
    "=\n",
    "1\n",
    "−\n",
    "6\n",
    "5\n",
    "y \n",
    "1\n",
    "​\n",
    " =1− \n",
    "5\n",
    "​\n",
    " \n",
    "6\n",
    "​\n",
    " \n",
    "y\n",
    "2\n",
    "=\n",
    "1\n",
    "−\n",
    "2\n",
    "5\n",
    "y \n",
    "2\n",
    "​\n",
    " =1− \n",
    "5\n",
    "​\n",
    " \n",
    "2\n",
    "​\n",
    " \n",
    "y\n",
    "3\n",
    "=\n",
    "1\n",
    "+\n",
    "2\n",
    "5\n",
    "y \n",
    "3\n",
    "​\n",
    " =1+ \n",
    "5\n",
    "​\n",
    " \n",
    "2\n",
    "​\n",
    " \n",
    "y\n",
    "4\n",
    "=\n",
    "1\n",
    "+\n",
    "6\n",
    "5\n",
    "y \n",
    "4\n",
    "​\n",
    " =1+ \n",
    "5\n",
    "​\n",
    " \n",
    "6\n",
    "​\n",
    " \n",
    "\n",
    "数值近似：\n",
    "y₁ ≈ -1.6833，y₂ ≈ 0.1056，y₃ ≈ 1.8944，y₄ ≈ 3.6833"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "6d2b1566",
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch.nn as nn\n",
    "\n",
    "class Residual(nn.Module):\n",
    "    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):\n",
    "        super().__init__()\n",
    "        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)\n",
    "        self.bn1 = nn.BatchNorm2d(out_channels)\n",
    "        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)\n",
    "        self.bn2 = nn.BatchNorm2d(out_channels)\n",
    "        self.relu = nn.ReLU(inplace=True)\n",
    "        \n",
    "        self.use_1x1conv = use_1x1conv\n",
    "        if use_1x1conv:\n",
    "            self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)\n",
    "            self.bn_shortcut = nn.BatchNorm2d(out_channels)\n",
    "        else:\n",
    "            self.conv_shortcut = None\n",
    "\n",
    "    def forward(self, x):\n",
    "        identity = x\n",
    "        out = self.relu(self.bn1(self.conv1(x)))\n",
    "        out = self.bn2(self.conv2(out))\n",
    "        if self.use_1x1conv:\n",
    "            identity = self.bn_shortcut(self.conv_shortcut(x))\n",
    "        out += identity\n",
    "        out = self.relu(out)\n",
    "        return out"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1088253c",
   "metadata": {},
   "source": [
    "为什么底层学习率小，顶层学习率大？\n",
    "底层特征（边缘、纹理等）在大型源数据集上已训练得很好，具有通用性。小学习率（或冻结）可以保留这些通用特征，避免在目标小数据集上过拟合或灾难性遗忘。顶层输出层需要学习新任务的类别区分，随机初始化或与源任务差异大，因此需要较大学习率快速适应。\n",
    "\n",
    "目标数据集很小且与源数据集非常相似时的策略\n",
    "\n",
    "冻结所有底层特征提取层（即前若干层），只训练顶层分类层（全连接层）。\n",
    "\n",
    "使用非常小的学习率进行微调，或者只重新训练最后一层（线性分类器）。\n",
    "\n",
    "可增加数据增广、权重衰减等正则化手段。"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c7722507",
   "metadata": {},
   "outputs": [
    {
     "ename": "ImportError",
     "evalue": "cannot import name 'OrderedDict' from 'typing' (c:\\Users\\12224\\Anaconda3\\lib\\typing.py)",
     "output_type": "error",
     "traceback": [
      "\u001b[1;31m---------------------------------------------------------------------------\u001b[0m",
      "\u001b[1;31mImportError\u001b[0m                               Traceback (most recent call last)",
      "\u001b[1;32m<ipython-input-1-00130c52690e>\u001b[0m in \u001b[0;36m<module>\u001b[1;34m()\u001b[0m\n\u001b[1;32m----> 1\u001b[1;33m \u001b[1;32mfrom\u001b[0m \u001b[0mtorchvision\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mtransforms\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0m\u001b[0;32m      2\u001b[0m \u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m      3\u001b[0m augmentation_pipeline = transforms.Compose([\n\u001b[0;32m      4\u001b[0m     \u001b[0mtransforms\u001b[0m\u001b[1;33m.\u001b[0m\u001b[0mRandomResizedCrop\u001b[0m\u001b[1;33m(\u001b[0m\u001b[1;36m224\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mscale\u001b[0m\u001b[1;33m=\u001b[0m\u001b[1;33m(\u001b[0m\u001b[1;36m0.08\u001b[0m\u001b[1;33m,\u001b[0m \u001b[1;36m1.0\u001b[0m\u001b[1;33m)\u001b[0m\u001b[1;33m)\u001b[0m\u001b[1;33m,\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m      5\u001b[0m     \u001b[0mtransforms\u001b[0m\u001b[1;33m.\u001b[0m\u001b[0mRandomHorizontalFlip\u001b[0m\u001b[1;33m(\u001b[0m\u001b[0mp\u001b[0m\u001b[1;33m=\u001b[0m\u001b[1;36m0.5\u001b[0m\u001b[1;33m)\u001b[0m\u001b[1;33m,\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n",
      "\u001b[1;32mc:\\Users\\12224\\Anaconda3\\lib\\site-packages\\torchvision\\__init__.py\u001b[0m in \u001b[0;36m<module>\u001b[1;34m()\u001b[0m\n\u001b[0;32m      3\u001b[0m \u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m      4\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mtorch\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[1;32m----> 5\u001b[1;33m \u001b[1;32mfrom\u001b[0m \u001b[0mtorchvision\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mdatasets\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mio\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mmodels\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mops\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mtransforms\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mutils\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0m\u001b[0;32m      6\u001b[0m \u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m      7\u001b[0m \u001b[1;32mfrom\u001b[0m \u001b[1;33m.\u001b[0m\u001b[0mextension\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0m_HAS_OPS\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n",
      "\u001b[1;32mc:\\Users\\12224\\Anaconda3\\lib\\site-packages\\torchvision\\models\\__init__.py\u001b[0m in \u001b[0;36m<module>\u001b[1;34m()\u001b[0m\n\u001b[0;32m     14\u001b[0m \u001b[1;32mfrom\u001b[0m \u001b[1;33m.\u001b[0m\u001b[0mvision_transformer\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[1;33m*\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m     15\u001b[0m \u001b[1;32mfrom\u001b[0m \u001b[1;33m.\u001b[0m\u001b[0mswin_transformer\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[1;33m*\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[1;32m---> 16\u001b[1;33m \u001b[1;32mfrom\u001b[0m \u001b[1;33m.\u001b[0m\u001b[0mmaxvit\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[1;33m*\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0m\u001b[0;32m     17\u001b[0m \u001b[1;32mfrom\u001b[0m \u001b[1;33m.\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mdetection\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0moptical_flow\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mquantization\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0msegmentation\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mvideo\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m     18\u001b[0m \u001b[1;32mfrom\u001b[0m \u001b[1;33m.\u001b[0m\u001b[0m_api\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mget_model\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mget_model_builder\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mget_model_weights\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mget_weight\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mlist_models\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n",
      "\u001b[1;32mc:\\Users\\12224\\Anaconda3\\lib\\site-packages\\torchvision\\models\\maxvit.py\u001b[0m in \u001b[0;36m<module>\u001b[1;34m()\u001b[0m\n\u001b[0;32m      1\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mmath\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m      2\u001b[0m \u001b[1;32mfrom\u001b[0m \u001b[0mfunctools\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mpartial\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[1;32m----> 3\u001b[1;33m \u001b[1;32mfrom\u001b[0m \u001b[0mtyping\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mAny\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mCallable\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mList\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mOptional\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mOrderedDict\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mSequence\u001b[0m\u001b[1;33m,\u001b[0m \u001b[0mTuple\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0m\u001b[0;32m      4\u001b[0m \u001b[1;33m\u001b[0m\u001b[0m\n\u001b[0;32m      5\u001b[0m \u001b[1;32mimport\u001b[0m \u001b[0mnumpy\u001b[0m \u001b[1;32mas\u001b[0m \u001b[0mnp\u001b[0m\u001b[1;33m\u001b[0m\u001b[0m\n",
      "\u001b[1;31mImportError\u001b[0m: cannot import name 'OrderedDict' from 'typing' (c:\\Users\\12224\\Anaconda3\\lib\\typing.py)"
     ]
    }
   ],
   "source": [
    "from torchvision import transforms\n",
    "\n",
    "augmentation_pipeline = transforms.Compose([\n",
    "    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),\n",
    "    transforms.RandomHorizontalFlip(p=0.5),\n",
    "    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),\n",
    "    transforms.ToTensor()\n",
    "])# 修复 typing.OrderedDict 导入问题\n",
    "import typing\n",
    "from collections import OrderedDict as _OD\n",
    "if not hasattr(typing, 'OrderedDict'):\n",
    "    typing.OrderedDict = _OD# 修复 typing.OrderedDict 导入问题\n",
    "import typing\n",
    "from collections import OrderedDict as _OD\n",
    "if not hasattr(typing, 'OrderedDict'):\n",
    "    typing.OrderedDict = _OD\n",
    "\n",
    "# 修复 typing.OrderedDict 导入问题\n",
    "import typing\n",
    "from collections import OrderedDict as _OD\n",
    "if not hasattr(typing, 'OrderedDict'):\n",
    "    typing.OrderedDict = _OD\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "794661a6",
   "metadata": {},
   "source": [
    "IoU 计算\n",
    "真实框 A = [10, 10, 50, 50]\n",
    "预测框 B = [30, 30, 70, 70]\n",
    "\n",
    "交集矩形：左上角 (max(10,30)=30, max(10,30)=30)，右下角 (min(50,70)=50, min(50,70)=50)\n",
    "交集面积 = (50-30) × (50-30) = 20×20 = 400\n",
    "\n",
    "A 面积 = (50-10)×(50-10) = 40×40 = 1600\n",
    "B 面积 = (70-30)×(70-30) = 40×40 = 1600\n",
    "并集面积 = 1600 + 1600 - 400 = 2800\n",
    "\n",
    "IoU = 400 / 2800 = 1/7（约 0.142857）"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4e2a4863",
   "metadata": {},
   "outputs": [
    {
     "ename": "SyntaxError",
     "evalue": "invalid syntax (410000643.py, line 17)",
     "output_type": "error",
     "traceback": [
      "  \u001b[36mCell\u001b[39m\u001b[36m \u001b[39m\u001b[32mIn[2]\u001b[39m\u001b[32m, line 17\u001b[39m\n\u001b[31m    \u001b[39m\u001b[31mreturn lossimport torch\u001b[39m\n                      ^\n\u001b[31mSyntaxError\u001b[39m\u001b[31m:\u001b[39m invalid syntax\n"
     ]
    }
   ],
   "source": [
    "import torch\n",
    "import torch.nn.functional as F\n",
    "\n",
    "def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):\n",
    "    \"\"\"\n",
    "    logits: 模型输出，形状 (N, K)\n",
    "    labels: 真实标签，形状 (N,)\n",
    "    epsilon: 平滑因子\n",
    "    \"\"\"\n",
    "    K = logits.size(-1)\n",
    "    # 构建平滑标签\n",
    "    smooth_labels = torch.full_like(logits, epsilon / (K - 1))\n",
    "    smooth_labels.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)\n",
    "    # 计算交叉熵\n",
    "    log_probs = F.log_softmax(logits, dim=-1)\n",
    "    loss = - (smooth_labels * log_probs).sum(dim=-1).mean()\n",
    "    return loss\n",
    "import torch\n",
    "import torch.nn.functional as F\n",
    "\n",
    "def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):\n",
    "    \"\"\"\n",
    "    logits: 模型输出，形状 (N, K)\n",
    "    labels: 真实标签，形状 (N,)\n",
    "    epsilon: 平滑因子\n",
    "    \"\"\"\n",
    "    K = logits.size(-1)\n",
    "    # 构建平滑标签\n",
    "    smooth_labels = torch.full_like(logits, epsilon / (K - 1))\n",
    "    smooth_labels.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)\n",
    "    # 计算交叉熵\n",
    "    log_probs = F.log_softmax(logits, dim=-1)\n",
    "    loss = - (smooth_labels * log_probs).sum(dim=-1).mean()\n",
    "    return loss\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "073aa27d",
   "metadata": {},
   "source": []
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9d5fdb6d",
   "metadata": {},
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "markdown",
   "id": "1d4e2a71",
   "metadata": {},
   "source": []
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "338b6dde",
   "metadata": {},
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "markdown",
   "id": "e3248cfc",
   "metadata": {},
   "source": []
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c2cf9281",
   "metadata": {},
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "markdown",
   "id": "31fe2d6b",
   "metadata": {},
   "source": []
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "358ee791",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "base",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined